# Evaluating attribution faithfulness with `autolrp.metrics`

An attribution is useful only if it identifies what the model uses. `autolrp.metrics` answers that with three numbers: **conservation** (did the relevance that reached the input add up to what was seeded?), and the **deletion / insertion** perturbation curves with their **AOPC** (does removing what the map calls relevant collapse the score, and does adding it back restore it?).

Every metric takes the explained tensor `x` (its `.relevance` is the attribution) and `score`, the number that was explained as a function of the input: the same expression that was seeded. To score another method's map on the same input, pass `R=`.

This notebook compares three maps for one `(model, input)` so the metrics have something to distinguish: **LRP-composite** (z⁺ on conv, ε elsewhere), a **Sobel edge filter** (a fixed image filter that never looks at the model), and **uniform random** (a lower bound).

In [ ]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autolrp (or `pip install -e .`)
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

import autolrp
from autolrp import LRPConfig, metrics
import _common  # noqa  (applies the showcase rcParams)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)

In [ ]:
# ResNet-18, ImageNet weights, a 128x128 crop so the per-step forwards stay fast on CPU.
model = resnet18(weights=ResNet18_Weights.DEFAULT).eval().to(device)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
preprocess = transforms.Compose([
    transforms.Resize((128, 128)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
x = preprocess(Image.open('../../../data/cat0.jpg').convert('RGB')).unsqueeze(0).to(device)
with torch.no_grad():
    target = int(model(x).argmax(-1))
with open('../../../data/imagenet_classes.txt') as f:
    classes = [c.strip() for c in f if c.strip()]
print(f'predicted: class {target} = {classes[target]}')

# The number we explain, as a function of the input: the predicted-class probability.
score = lambda t: torch.softmax(model(t)[0], -1)[target]

## Three maps for the same input

`R_lrp` comes from `.lrp()`; the other two are per-pixel `(1, 1, H, W)` maps and are broadcast over the channels by the metrics.

In [ ]:
# 1. LRP-composite.
xt = autolrp.tensor(x.clone())
model(xt)[0, target].lrp(config=LRPConfig.composite())
R_lrp = xt.relevance.detach().sum(dim=1, keepdim=True)

# 2. Sobel edge magnitude: never sees the model.
gray = x.mean(dim=1, keepdim=True)
sx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=x.dtype, device=device).view(1, 1, 3, 3)
R_sobel = (F.conv2d(gray, sx, padding=1) ** 2 + F.conv2d(gray, sx.transpose(2, 3), padding=1) ** 2).sqrt()

# 3. Uniform random: the lower bound.
R_random = torch.randn(1, 1, 128, 128, generator=torch.Generator(device=device).manual_seed(42), device=device)

METHODS = {'LRP-composite': R_lrp, 'Sobel-edge': R_sobel, 'random': R_random}

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
img = (x[0].cpu() * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)).clamp(0, 1).permute(1, 2, 0).numpy()
axes[0].imshow(img); axes[0].set_title('input'); axes[0].axis('off')
for ax, (name, R) in zip(axes[1:], METHODS.items()):
    m = R[0, 0].detach().cpu().numpy(); v = abs(m).max() + 1e-9
    ax.imshow(m, cmap='coolwarm', vmin=-v, vmax=v); ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()

## Conservation

The engine seeds `+1`, so a conserving attribution sums to `1.0` at the input; the deficit is what the biases absorbed. It applies only to the LRP map (the other two were not propagated), and it is a gate, not a score: a uniform map conserves too.

In [ ]:
print(f'conservation of R_lrp: {metrics.conservation(xt):.3f}')

## Deletion and insertion curves (Petsiuk et al. 2018)

Positions are ranked by `|R|`. **Deletion** replaces the most relevant ones by the per-channel mean and watches the probability collapse; **insertion** starts from that mean image and restores them. A faithful map drops early in deletion and recovers early in insertion; a map that ranks the wrong positions gives flat or diagonal curves.

In [ ]:
curves = {name: {mode: metrics.perturbation_curve(xt, score, R=R, mode=mode, n_steps=20, baseline='mean')
                 for mode in ('deletion', 'insertion')} for name, R in METHODS.items()}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = {'LRP-composite': '#cc3333', 'Sobel-edge': '#3b6fb6', 'random': '#888888'}
for ax, mode, title in [(axes[0], 'deletion', 'Deletion: remove top-|R| first (lower is better)'),
                        (axes[1], 'insertion', 'Insertion: restore top-|R| first (higher is better)')]:
    for name in METHODS:
        c = curves[name][mode]
        ax.plot(c['fractions'], c['scores'], '-o', label=name, color=colors[name], markersize=3)
    ax.set_xlabel('fraction of input perturbed'); ax.set_ylabel('predicted-class probability')
    ax.set_title(title, fontsize=10); ax.legend(fontsize=9, frameon=False)
plt.tight_layout(); plt.show()

## AOPC (Samek et al. 2017)

Each curve as one number, larger is better in both modes: the mean drop from the intact input (deletion) and the mean gain over the baseline (insertion).

In [ ]:
print(f'{"method":16s}  {"AOPC deletion":>14s}  {"AOPC insertion":>15s}')
for name, R in METHODS.items():
    d = metrics.aopc(xt, score, R=R, mode='deletion',  n_steps=20, baseline='mean')
    i = metrics.aopc(xt, score, R=R, mode='insertion', n_steps=20, baseline='mean')
    print(f'{name:16s}  {d:+14.4f}  {i:+15.4f}')

## All three in one call

`faithfulness` returns conservation and both AOPCs for the map on `x`; this is the check to run on any new configuration before looking at heatmaps.

In [ ]:
metrics.faithfulness(xt, score, n_steps=20, baseline='mean')

## Reading them together

The perturbation metrics judge the *ranking* of `|R|` against what the model actually uses. On natural images an edge filter can score respectably here, not because it is faithful but because edges are where ImageNet models look; a map that scores well without ever seeing the model is a feature detector, not an explanation. Conservation is the complementary check: it says nothing about ranking, but it fails the moment a rule is wrong. An attribution that conserves, collapses early under deletion and recovers early under insertion is faithful in the ordinary sense of the word.